# 02 · Advanced encrypted analytics: products, statistics, SUMIF, ML scoring, time series

Builds on the data loaded by notebook 01. CKKS ciphertexts here have **two multiplicative levels**
(`q = [60, 35, 35, 60]`, scale 2^35), enough for products of three encrypted values or a degree-3
polynomial. No relinearization key is uploaded: products stay as 3- or 4-polynomial ciphertexts, which
Snowflake can still add up and the laptop can still decrypt.

In [1]:
import numpy as np
import json

import pandas as pd

from he_snowflake import data
from he_snowflake.config import get_session, load_config
from he_snowflake.keys import load_all
from he_snowflake.verify import Checks, run_encrypted

pd.set_option("display.float_format", "{:,.6f}".format)
cfg = load_config()
session = get_session(cfg)
keys = load_all(cfg.key_dir, max_packed_values=cfg.n_employees)
BFV, CKKS = keys["bfv"], keys["ckks"]
emp = data.employees(cfg.n_employees, cfg.seed)
rev = data.monthly_revenue(cfg.seed)
checks = Checks()
CKKS_TOL = 1e-5   # relative, depth <= 2

## 1. Encrypted × encrypted: labor cost, SUMPRODUCT, hours-weighted average rate
`labor = rate × hours` per row (both encrypted); the weighted average `Σ(rate·hours) / Σ hours` has an
encrypted numerator *and* an encrypted denominator.

In [2]:
SQL = """
SELECT e.region,
       HE_SUM(k.context, HE_MUL(k.context, e.hourly_rate_ct, e.hours_ct)) AS labor_cost,
       HE_SUM(k.context, e.hours_ct)                                      AS hours
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.ckks_key_id
GROUP BY e.region ORDER BY e.region"""
lab, secs = run_encrypted(session, SQL, {"LABOR_COST": CKKS, "HOURS": CKKS})
lab = lab.set_index("REGION")
lab["weighted avg rate"] = lab.LABOR_COST / lab.HOURS
grp = emp.assign(l=emp.hourly_rate * emp.hours_logged).groupby("region")
lab["labor (plaintext)"] = grp.l.sum()
lab["weighted avg rate (plaintext)"] = grp.l.sum() / grp.hours_logged.sum()
checks.frame("labor cost = SUM(rate*hours)", lab, "LABOR_COST", "labor (plaintext)", rel_tol=CKKS_TOL)
checks.frame("hours-weighted avg rate", lab, "weighted avg rate", "weighted avg rate (plaintext)", rel_tol=CKKS_TOL)
print(f"query {secs:.1f}s")
lab

OK       labor cost = SUM(rate*hours) [AMER]: decrypted=159,658.144602 plaintext=159,658.144600 abs_err=2.40e-06
OK       labor cost = SUM(rate*hours) [APJ]: decrypted=56,198.142398 plaintext=56,198.142400 abs_err=1.99e-06
OK       labor cost = SUM(rate*hours) [EMEA]: decrypted=126,210.625188 plaintext=126,210.625200 abs_err=1.24e-05
OK       hours-weighted avg rate [AMER]: decrypted=94.269790 plaintext=94.269790 abs_err=2.12e-10
OK       hours-weighted avg rate [APJ]: decrypted=102.904384 plaintext=102.904384 abs_err=1.24e-08
OK       hours-weighted avg rate [EMEA]: decrypted=95.479571 plaintext=95.479571 abs_err=5.22e-09
query 7.4s


,LABOR_COST,HOURS,weighted avg rate,labor (plaintext),weighted avg rate (plaintext)
REGION,,,,,
AMER,"159,658.144602","1,693.630000",94.269790,"159,658.144600",94.269790
APJ,"56,198.142398",546.120000,102.904384,"56,198.142400",102.904384
EMEA,"126,210.625188","1,321.860000",95.479571,"126,210.625200",95.479571


## 2. Statistics from encrypted sufficient statistics
Snowflake returns encrypted `Σx, Σx², Σy, Σy², Σxy` (x = hourly rate, y = performance score). Everything
else is plaintext algebra on the decrypted sums: mean, variance, standard deviation, covariance, Pearson
correlation, and the least-squares line `perf ≈ a + b·rate`.

In [3]:
SQL = """
SELECT COUNT(*) AS n,
       HE_SUM(k.context, e.hourly_rate_ct)                              AS sx,
       HE_SUM(k.context, HE_SQUARE(k.context, e.hourly_rate_ct))        AS sxx,
       HE_SUM(k.context, e.perf_score_ct)                               AS sy,
       HE_SUM(k.context, HE_SQUARE(k.context, e.perf_score_ct))         AS syy,
       HE_SUM(k.context, HE_MUL(k.context, e.hourly_rate_ct, e.perf_score_ct)) AS sxy
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.ckks_key_id"""
s, secs = run_encrypted(session, SQL, {c: CKKS for c in ("SX", "SXX", "SY", "SYY", "SXY")})
s = s.iloc[0]
n = s.N
var_x, var_y = s.SXX / n - (s.SX / n) ** 2, s.SYY / n - (s.SY / n) ** 2
cov = s.SXY / n - (s.SX / n) * (s.SY / n)
slope = cov / var_x
stats = {
    "mean(rate)": (s.SX / n, emp.hourly_rate.mean()),
    "var_pop(rate)": (var_x, emp.hourly_rate.var(ddof=0)),
    "var_samp(rate)": (var_x * n / (n - 1), emp.hourly_rate.var(ddof=1)),
    "stddev(rate)": (np.sqrt(var_x), emp.hourly_rate.std(ddof=0)),
    "covar_pop(rate, perf)": (cov, np.cov(emp.hourly_rate, emp.perf_score, ddof=0)[0, 1]),
    "corr(rate, perf)": (cov / np.sqrt(var_x * var_y), emp.hourly_rate.corr(emp.perf_score)),
    "regr_slope(perf ~ rate)": (slope, np.polyfit(emp.hourly_rate, emp.perf_score, 1)[0]),
    "regr_intercept(perf ~ rate)": (s.SY / n - slope * s.SX / n, np.polyfit(emp.hourly_rate, emp.perf_score, 1)[1]),
}
for label, (got, exp) in stats.items():
    checks.check(label, got, exp, rel_tol=1e-4)
print(f"query {secs:.1f}s")

OK       mean(rate): decrypted=98.194100 plaintext=98.194100 abs_err=2.88e-09
OK       var_pop(rate): decrypted=2,236.159638 plaintext=2,236.159638 abs_err=1.28e-07
OK       var_samp(rate): decrypted=2,258.747109 plaintext=2,258.747109 abs_err=1.29e-07
OK       stddev(rate): decrypted=47.288050 plaintext=47.288050 abs_err=1.35e-09
OK       covar_pop(rate, perf): decrypted=35.129381 plaintext=35.129381 abs_err=2.12e-07
OK       corr(rate, perf): decrypted=0.895640 plaintext=0.895640 abs_err=4.92e-08
OK       regr_slope(perf ~ rate): decrypted=0.015710 plaintext=0.015710 abs_err=9.40e-11
OK       regr_intercept(perf ~ rate): decrypted=1.421601 plaintext=1.421601 abs_err=1.24e-08
query 10.5s


## 3. Encrypted SUMIF / COUNTIF: the condition itself is secret
`remote_flag` is an **encrypted** 0/1 column. Multiplying by it keeps qualifying rows and zeroes the others,
and Snowflake cannot tell which rows qualified. `office = 1 − flag` is computed homomorphically too.

* COUNTIF(remote) = Σ flag
* SUMIF(remote, rate) = Σ flag·rate  → average remote rate
* SUMIF(remote, rate·hours) = Σ flag·rate·hours  (three encrypted factors, depth 2)

In [4]:
SQL = """
SELECT e.department,
       HE_SUM(k.context, e.remote_flag_ct)                                         AS remote_n,
       HE_SUM(k.context, HE_MUL(k.context, e.remote_flag_ct, e.hourly_rate_ct))    AS remote_rate_sum,
       HE_SUM(k.context, HE_MUL(k.context, HE_MUL(k.context, e.hourly_rate_ct, e.hours_ct), e.remote_flag_ct)) AS remote_labor,
       HE_SUM(k.context, HE_MUL(k.context, HE_ADD_PLAIN(k.context, HE_NEG(k.context, e.remote_flag_ct), 1),
                                           e.hourly_rate_ct))                      AS office_rate_sum
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.ckks_key_id
GROUP BY e.department ORDER BY e.department"""
sif, secs = run_encrypted(session, SQL, {c: CKKS for c in ("REMOTE_N", "REMOTE_RATE_SUM", "REMOTE_LABOR", "OFFICE_RATE_SUM")})
sif = sif.set_index("DEPARTMENT")
g = emp.assign(r=emp.remote_flag, rr=emp.remote_flag * emp.hourly_rate,
               rl=emp.remote_flag * emp.hourly_rate * emp.hours_logged,
               orr=(1 - emp.remote_flag) * emp.hourly_rate).groupby("department")
sif["avg remote rate"] = sif.REMOTE_RATE_SUM / sif.REMOTE_N
checks.frame("COUNTIF(remote)", sif.assign(t=g.r.sum()), "REMOTE_N", "t", abs_tol=1e-4)
checks.frame("SUMIF(remote, rate)", sif.assign(t=g.rr.sum()), "REMOTE_RATE_SUM", "t", rel_tol=CKKS_TOL, abs_tol=1e-3)
checks.frame("SUMIF(remote, rate*hours)", sif.assign(t=g.rl.sum()), "REMOTE_LABOR", "t", rel_tol=CKKS_TOL, abs_tol=1e-2)
checks.frame("SUMIF(office, rate)", sif.assign(t=g.orr.sum()), "OFFICE_RATE_SUM", "t", rel_tol=CKKS_TOL, abs_tol=1e-3)
print(f"query {secs:.1f}s")
sif

OK       COUNTIF(remote) [Engineering]: decrypted=9.000000 plaintext=9.000000 abs_err=1.38e-08
OK       COUNTIF(remote) [Finance]: decrypted=8.000000 plaintext=8.000000 abs_err=3.50e-08
OK       COUNTIF(remote) [Marketing]: decrypted=8.000000 plaintext=8.000000 abs_err=4.00e-10
OK       COUNTIF(remote) [Sales]: decrypted=6.000000 plaintext=6.000000 abs_err=1.43e-07
OK       COUNTIF(remote) [Support]: decrypted=3.000000 plaintext=3.000000 abs_err=6.95e-08
OK       SUMIF(remote, rate) [Engineering]: decrypted=1,049.959996 plaintext=1,049.960000 abs_err=4.30e-06
OK       SUMIF(remote, rate) [Finance]: decrypted=535.679992 plaintext=535.680000 abs_err=8.28e-06
OK       SUMIF(remote, rate) [Marketing]: decrypted=1,036.699999 plaintext=1,036.700000 abs_err=1.45e-06
OK       SUMIF(remote, rate) [Sales]: decrypted=793.469981 plaintext=793.470000 abs_err=1.93e-05
OK       SUMIF(remote, rate) [Support]: decrypted=287.790005 plaintext=287.790000 abs_err=5.23e-06
OK       SUMIF(remote, rate*hours)

,REMOTE_N,REMOTE_RATE_SUM,REMOTE_LABOR,OFFICE_RATE_SUM,avg remote rate
DEPARTMENT,,,,,
Engineering,9.000000,"1,049.959996","34,507.707195","2,293.040004",116.662222
Finance,8.000000,535.679992,"19,928.911489",872.420008,66.959999
Marketing,8.000000,"1,036.699999","39,074.275133",874.700002,129.587500
Sales,6.000000,793.469981,"15,942.778264","1,536.490019",132.245000
Support,3.000000,287.790005,"11,131.297125",539.159995,95.930000


## 4. ML scoring on encrypted features
**Linear model** with plaintext weights (the model owner's weights could equally be encrypted):
`score = 0.4·perf + 0.05·tenure + 0.002·rate − 1.5`. Per-row encrypted scores come back, and the laptop
decrypts and ranks them.

**Logistic risk** via a degree-3 sigmoid approximation, σ(z) ≈ 0.5 + 0.197z − 0.004z³ on z ∈ [−4, 4], with
z = 2·(3 − perf). The affine map is folded into the polynomial's coefficients on the client, so Snowflake
evaluates a single cubic in the encrypted `perf` (`HE_POLY`, depth 2).

In [5]:
W = {"perf": 0.4, "tenure": 0.05, "rate": 0.002, "bias": -1.5}
SQL = f"""
SELECT e.emp_id,
       HE_ADD_PLAIN(k.context,
           HE_ADD(k.context,
               HE_ADD(k.context, HE_MUL_PLAIN(k.context, e.perf_score_ct, {W['perf']}),
                                 HE_MUL_PLAIN(k.context, e.tenure_ct, {W['tenure']})),
               HE_MUL_PLAIN(k.context, e.hourly_rate_ct, {W['rate']})),
           {W['bias']}) AS score
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.ckks_key_id ORDER BY e.emp_id"""
sc, secs = run_encrypted(session, SQL, {"SCORE": CKKS})
plain_score = W["perf"] * emp.perf_score + W["tenure"] * emp.tenure_years + W["rate"] * emp.hourly_rate + W["bias"]
sc["plaintext"] = plain_score.values
checks.check("linear score, max abs error over all rows", (sc.SCORE - sc.plaintext).abs().max(), 0.0, abs_tol=1e-4)
top = sc.nlargest(5, "SCORE")
print(f"query {secs:.1f}s | top-5 by decrypted score == top-5 by plaintext score:",
      list(top.EMP_ID) == list(sc.nlargest(5, "plaintext").EMP_ID))
top

OK       linear score, max abs error over all rows: decrypted=0.000000 plaintext=0.000000 abs_err=7.47e-08
query 11.0s | top-5 by decrypted score == top-5 by plaintext score: True


,EMP_ID,SCORE,plaintext
96,97,1.463660,1.463660
64,65,1.263500,1.263500
32,33,1.161540,1.161540
25,26,1.129700,1.129700
45,46,1.127000,1.127000


In [6]:
# sigmoid(z) ~ c0 + c1 z + c3 z^3 with z = A*perf + B  ->  expand into a cubic in perf (client side)
c0, c1, c3 = 0.5, 0.197, -0.004
A, B = -2.0, 6.0
poly = np.polynomial.Polynomial([c0, c1, 0.0, c3])(np.polynomial.Polynomial([B, A]))
coeffs = [float(c) for c in poly.coef]
print("coefficients sent to HE_POLY:", [round(c, 6) for c in coeffs])
SQL = """
SELECT e.department, COUNT(*) AS n,
       HE_SUM(k.context, HE_POLY(k.context, e.perf_score_ct, PARSE_JSON(?)::ARRAY)) AS risk_sum
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.ckks_key_id
GROUP BY e.department ORDER BY e.department"""
risk, secs = run_encrypted(session, SQL, {"RISK_SUM": CKKS}, params=[str(coeffs)])
risk = risk.set_index("DEPARTMENT")
risk["avg risk (approx sigmoid)"] = risk.RISK_SUM / risk.N
z = A * emp.perf_score + B
risk["avg risk (same poly, plaintext)"] = (c0 + c1 * z + c3 * z ** 3).groupby(emp.department).mean()
risk["avg risk (true sigmoid)"] = (1 / (1 + np.exp(-z))).groupby(emp.department).mean()
checks.frame("avg attrition risk (HE_POLY)", risk, "avg risk (approx sigmoid)", "avg risk (same poly, plaintext)", abs_tol=2e-3)
print(f"query {secs:.1f}s")
risk

coefficients sent to HE_POLY: [0.818, 0.47, -0.288, 0.032]


OK       avg attrition risk (HE_POLY) [Engineering]: decrypted=0.527388 plaintext=0.527384 abs_err=3.84e-06
OK       avg attrition risk (HE_POLY) [Finance]: decrypted=0.651626 plaintext=0.651624 abs_err=2.41e-06
OK       avg attrition risk (HE_POLY) [Marketing]: decrypted=0.441943 plaintext=0.441947 abs_err=3.67e-06
OK       avg attrition risk (HE_POLY) [Sales]: decrypted=0.415443 plaintext=0.415440 abs_err=3.50e-06
OK       avg attrition risk (HE_POLY) [Support]: decrypted=0.538869 plaintext=0.538871 abs_err=2.26e-06
query 5.4s


,N,RISK_SUM,avg risk (approx sigmoid),"avg risk (same poly, plaintext)",avg risk (true sigmoid)
DEPARTMENT,,,,,
Engineering,33,17.403805,0.527388,0.527384,0.525612
Finance,19,12.380900,0.651626,0.651624,0.659082
Marketing,19,8.396925,0.441943,0.441947,0.441349
Sales,21,8.724307,0.415443,0.415440,0.413002
Support,8,4.310951,0.538869,0.538871,0.539046


## 5. Time series: running totals, month-over-month deltas, moving sums
* **YTD revenue** per region with the `HE_RUNNING_SUM` UDTF (`PARTITION BY region ORDER BY month`)
* **MoM delta** = `HE_SUB(revenue, LAG(revenue))`: `LAG` moves ciphertexts between rows; Snowflake never sees values
* **3-month moving sum** = `revenue + LAG(1) + LAG(2)`

In [7]:
SQL = """
SELECT r.region, t.row_key AS month, t.running_ct AS ytd
FROM REVENUE_MONTHLY_ENC r JOIN HE_KEYS k ON k.key_id = r.bfv_key_id,
     TABLE(HE_RUNNING_SUM(k.context, r.revenue_ct, TO_VARCHAR(r.month))
           OVER (PARTITION BY r.region ORDER BY r.month)) t
ORDER BY r.region, month"""
ytd, secs_ytd = run_encrypted(session, SQL, {"YTD": BFV})

SQL = """
WITH x AS (
  SELECT r.region, r.month, k.context AS ctx, r.revenue_ct AS cur,
         LAG(r.revenue_ct, 1) OVER (PARTITION BY r.region ORDER BY r.month) AS prev1,
         LAG(r.revenue_ct, 2) OVER (PARTITION BY r.region ORDER BY r.month) AS prev2
  FROM REVENUE_MONTHLY_ENC r JOIN HE_KEYS k ON k.key_id = r.bfv_key_id)
SELECT region, TO_VARCHAR(month) AS month, cur AS revenue,
       HE_SUB(ctx, cur, prev1)                      AS mom_delta,
       HE_ADD(ctx, HE_ADD(ctx, cur, prev1), prev2)  AS moving_3m
FROM x ORDER BY region, month"""
ts, secs = run_encrypted(session, SQL, {"REVENUE": BFV, "MOM_DELTA": BFV, "MOVING_3M": BFV})
ts = ts.merge(ytd, on=["REGION", "MONTH"])
ts["MoM %"] = ts.MOM_DELTA / (ts.REVENUE - ts.MOM_DELTA) * 100

p = rev.assign(MONTH=rev.month.astype(str), REGION=rev.region).sort_values(["REGION", "MONTH"])
g = p.groupby("REGION").revenue_usd
p["ytd"], p["mom"], p["m3"] = g.cumsum(), g.diff(), g.rolling(3).sum().reset_index(level=0, drop=True)
ts = ts.merge(p[["REGION", "MONTH", "ytd", "mom", "m3"]], on=["REGION", "MONTH"])
checks.frame("YTD revenue (UDTF)", ts.set_index(["REGION", "MONTH"]), "YTD", "ytd")
checks.frame("MoM delta (LAG)", ts.dropna(subset=["mom"]).set_index(["REGION", "MONTH"]), "MOM_DELTA", "mom")
checks.frame("3-month moving sum", ts.dropna(subset=["m3"]).set_index(["REGION", "MONTH"]), "MOVING_3M", "m3")
print(f"queries {secs_ytd:.1f}s + {secs:.1f}s")
ts[ts.REGION == "EMEA"][["MONTH", "REVENUE", "YTD", "MOM_DELTA", "MoM %", "MOVING_3M"]]

OK       YTD revenue (UDTF) [('AMER', '2026-01-01')]: decrypted=3,821,372.000000 plaintext=3,821,372.000000 abs_err=0.00e+00
OK       YTD revenue (UDTF) [('AMER', '2026-02-01')]: decrypted=7,174,642.000000 plaintext=7,174,642.000000 abs_err=0.00e+00
OK       YTD revenue (UDTF) [('AMER', '2026-03-01')]: decrypted=10,769,879.000000 plaintext=10,769,879.000000 abs_err=0.00e+00
OK       YTD revenue (UDTF) [('AMER', '2026-04-01')]: decrypted=15,744,342.000000 plaintext=15,744,342.000000 abs_err=0.00e+00
OK       YTD revenue (UDTF) [('AMER', '2026-05-01')]: decrypted=20,614,001.000000 plaintext=20,614,001.000000 abs_err=0.00e+00
OK       YTD revenue (UDTF) [('AMER', '2026-06-01')]: decrypted=25,045,098.000000 plaintext=25,045,098.000000 abs_err=0.00e+00
OK       YTD revenue (UDTF) [('AMER', '2026-07-01')]: decrypted=30,149,165.000000 plaintext=30,149,165.000000 abs_err=0.00e+00
OK       YTD revenue (UDTF) [('AMER', '2026-08-01')]: decrypted=34,344,086.000000 plaintext=34,344,086.000000 abs_e

,MONTH,REVENUE,YTD,MOM_DELTA,MoM %,MOVING_3M
24,2026-01-01,2112013,2112013,NaN,NaN,NaN
25,2026-02-01,2245212,4357225,"133,199.000000",6.306732,NaN
26,2026-03-01,2539159,6896384,"293,947.000000",13.092171,"6,896,384.000000"
27,2026-04-01,2379169,9275553,"-159,990.000000",-6.300905,"7,163,540.000000"
28,2026-05-01,2769349,12044902,"390,180.000000",16.399844,"7,687,677.000000"
29,2026-06-01,3395223,15440125,"625,874.000000",22.600041,"8,543,741.000000"
30,2026-07-01,3235818,18675943,"-159,405.000000",-4.694979,"9,400,390.000000"
31,2026-08-01,2824455,21500398,"-411,363.000000",-12.712798,"9,455,496.000000"
32,2026-09-01,2588681,24089079,"-235,774.000000",-8.347593,"8,648,954.000000"
33,2026-10-01,2726158,26815237,"137,477.000000",5.310697,"8,139,294.000000"


## 6. What the server can learn: structure, never values
`HE_INFO` exposes what any server-side code can see: ciphertext size, polynomial count, level and scale. It
depends on *which operations ran*, not on the data.

In [8]:
info = session.sql("""
SELECT 1 AS step, 'fresh rate'                AS expr, HE_INFO(k.context, e.hourly_rate_ct) AS info
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.ckks_key_id WHERE e.emp_id = 1
UNION ALL
SELECT 2, 'rate * 1.05', HE_INFO(k.context, HE_MUL_PLAIN(k.context, e.hourly_rate_ct, 1.05))
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.ckks_key_id WHERE e.emp_id = 1
UNION ALL
SELECT 3, 'rate * hours', HE_INFO(k.context, HE_MUL(k.context, e.hourly_rate_ct, e.hours_ct))
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.ckks_key_id WHERE e.emp_id = 1
UNION ALL
SELECT 4, 'rate * hours * flag', HE_INFO(k.context, HE_MUL(k.context, HE_MUL(k.context, e.hourly_rate_ct, e.hours_ct), e.remote_flag_ct))
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.ckks_key_id WHERE e.emp_id = 1
UNION ALL
SELECT 5, 'cubic poly(perf)', HE_INFO(k.context, HE_POLY(k.context, e.perf_score_ct, [0.5, 0.1, 0.01, 0.001]))
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.ckks_key_id WHERE e.emp_id = 1
ORDER BY step
""").to_pandas()
pd.json_normalize(info.INFO.map(json.loads)).set_index(info.EXPR)[["polys", "level", "levels_left", "scale_bits", "bytes"]]

,polys,level,levels_left,scale_bits,bytes
EXPR,,,,,
fresh rate,2,0,2,35.000000,312415
rate * 1.05,2,1,1,35.000000,224194
rate * hours,3,1,1,35.000000,341189
rate * hours * flag,4,2,0,35.000000,262275
cubic poly(perf),4,2,0,35.000000,262275


In [9]:
summary = checks.summary()
summary[["check", "decrypted", "plaintext", "rel_err", "ok"]]

139/139 checks passed


,check,decrypted,plaintext,rel_err,ok
0,labor cost = SUM(rate*hours) [AMER],"159,658.144602","159,658.144600",0.000000,True
1,labor cost = SUM(rate*hours) [APJ],"56,198.142398","56,198.142400",0.000000,True
2,labor cost = SUM(rate*hours) [EMEA],"126,210.625188","126,210.625200",0.000000,True
3,hours-weighted avg rate [AMER],94.269790,94.269790,0.000000,True
4,hours-weighted avg rate [APJ],102.904384,102.904384,0.000000,True
...,...,...,...,...,...
134,"3-month moving sum [('EMEA', '2026-08-01')]","9,455,496.000000","9,455,496.000000",0.000000,True
135,"3-month moving sum [('EMEA', '2026-09-01')]","8,648,954.000000","8,648,954.000000",0.000000,True
136,"3-month moving sum [('EMEA', '2026-10-01')]","8,139,294.000000","8,139,294.000000",0.000000,True
137,"3-month moving sum [('EMEA', '2026-11-01')]","7,640,324.000000","7,640,324.000000",0.000000,True
